In [1]:
from vnstock import Listing, Quote
import pandas as pd
import time

In [2]:
listing = Listing(source="KBS")
vn30_symbols = listing.symbols_by_group(group_name="VN30")
vn30_symbols = vn30_symbols.to_list()
print(vn30_symbols)

['ACB', 'BID', 'CTG', 'DGC', 'FPT', 'GAS', 'GVR', 'HDB', 'HPG', 'LPB', 'MBB', 'MSN', 'MWG', 'PLX', 'SAB', 'SHB', 'SSB', 'SSI', 'STB', 'TCB', 'TPB', 'VCB', 'VHM', 'VIB', 'VIC', 'VJC', 'VNM', 'VPB', 'VPL', 'VRE']


In [3]:
dfs = []

success_count = 0
fail_count = 0

total = len(vn30_symbols)

for i, symbol in enumerate(vn30_symbols, 1):
    for attempt in range(3):
        try:
            start_time = time.time()

            quote = Quote(symbol=symbol, source='vci')
            df = quote.history(start='2017-01-01', end='2025-12-31', interval="1D")

            duration = time.time() - start_time

            if df is not None and not df.empty:
                df["symbol"] = symbol
                dfs.append(df)
                success_count += 1

            print(f"[{i}/{total}] {symbol} OK in {duration:.2f}s | success={success_count}")

            time.sleep(5)
            break

        except Exception as e:
            if attempt == 2:
                fail_count += 1
                print(f"[{i}/{total}] {symbol} FAILED after 3 attempts | fail={fail_count} | error={e}")
            else:
                print(f"{symbol} retry {attempt+1}/3 error: {e}")
                time.sleep(10)

[1/30] ACB OK in 0.60s | success=1
[2/30] BID OK in 0.66s | success=2
[3/30] CTG OK in 0.73s | success=3
[4/30] DGC OK in 0.76s | success=4
[5/30] FPT OK in 0.57s | success=5
[6/30] GAS OK in 0.78s | success=6
[7/30] GVR OK in 0.61s | success=7
[8/30] HDB OK in 0.72s | success=8
[9/30] HPG OK in 0.50s | success=9
[10/30] LPB OK in 0.79s | success=10
[11/30] MBB OK in 0.57s | success=11
[12/30] MSN OK in 0.83s | success=12
[13/30] MWG OK in 0.71s | success=13
[14/30] PLX OK in 0.56s | success=14
[15/30] SAB OK in 0.57s | success=15
[16/30] SHB OK in 0.57s | success=16
[17/30] SSB OK in 0.65s | success=17
[18/30] SSI OK in 0.57s | success=18
[19/30] STB OK in 0.60s | success=19
[20/30] TCB OK in 0.61s | success=20
[21/30] TPB OK in 0.57s | success=21
[22/30] VCB OK in 0.62s | success=22
[23/30] VHM OK in 0.47s | success=23
[24/30] VIB OK in 0.45s | success=24
[25/30] VIC OK in 0.60s | success=25
[26/30] VJC OK in 0.57s | success=26
[27/30] VNM OK in 0.69s | success=27
[28/30] VPB OK in 0

In [4]:
final_df = pd.concat(dfs, ignore_index=True)

start = '2017-01-01'
end = '2025-12-31'

final_df['time'] = pd.to_datetime(final_df['time'])
final_df = final_df[
    (final_df['time'] >= start) & 
    (final_df['time'] <= end)
]

final_df = final_df.sort_values(by=['symbol', 'time']).reset_index(drop=True)

print(final_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62196 entries, 0 to 62195
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   time    62196 non-null  datetime64[ns]
 1   open    62196 non-null  float64       
 2   high    62196 non-null  float64       
 3   low     62196 non-null  float64       
 4   close   62196 non-null  float64       
 5   volume  62196 non-null  int64         
 6   symbol  62196 non-null  object        
dtypes: datetime64[ns](1), float64(4), int64(1), object(1)
memory usage: 3.3+ MB
None


In [5]:
len(final_df.symbol.unique())

30

In [ ]:
cols = ['symbol'] + [col for col in final_df.columns if col != 'symbol']
df = final_df[cols]

df.to_csv('./data/vn30_from_2017_to_2025.csv', index=False)